# CSI-based 5G UE localization: ViT vs CNN, amplitude-phase vs real-imaginary input
DeepMIMO O1 3.5 GHz, gNB t006, 30k users, CSI = 32 antennas x 32 subcarriers.

**What this notebook does**
1. Spatial train / val / test split (10 m blocks of street).
2. Two input representations of the complex CSI, everything else identical:
   - `amp_phase`: |H| and phase (the representation asked for in the subject),
   - `re_im`: real and imaginary parts.
3. Trains a CNN baseline and a ViT for each representation, with several seeds (mean +- std).
4. Evaluates: mean / median / P90 / RMSE, CDF, learning curves, error map, outliers,
   robustness to noise, training-set size, and a CNN learning-rate check.

**Colab:** Runtime > Change runtime type > T4 GPU. Put `csi_o1.mat` in Google Drive and set `DATA_PATH`.
Expected runtime on a T4: about 45-60 min with 3 seeds (`FAST = True` for a 2-minute crash test).

## 1. Setup and settings

In [3]:
import os, sys, time, json
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.io import loadmat

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    DATA_PATH = '/content/csi_o1.mat'   # <- adjust
    OUT_DIR   = '/content/csi_results'  # figures, results.json, models.pt
else:
    DATA_PATH = 'csi_o1.mat'
    OUT_DIR   = 'results'
FIG_DIR = os.path.join(OUT_DIR, 'figs')
os.makedirs(FIG_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| outputs ->', OUT_DIR)

REPS        = ['amp_phase', 're_im']   # input representations compared
REP_LABEL   = {'amp_phase': 'Amp/Phase', 're_im': 'Re/Im'}
MODELS      = ['CNN', 'ViT']
SEEDS       = [0, 1, 2]                # repeated runs -> mean +- std
BLOCK_M     = 10                       # spatial split block size (m)
EPOCHS      = {'CNN': 40, 'ViT': 60}
LR          = {'CNN': 2e-3, 'ViT': 1e-3}
BATCH       = 256
RUN_DATA_SIZE   = True                 # retrain on 25 / 50 % of the training set
RUN_CNN_LR_CHECK = True                # CNN retrained with a lower learning rate
FAST        = False                    # True: few epochs, one seed (quick end-to-end check)
if FAST:
    EPOCHS, SEEDS = {'CNN': 2, 'ViT': 2}, [0]

def seed_all(s):
    torch.manual_seed(s); np.random.seed(s)
    if device == 'cuda': torch.cuda.manual_seed_all(s)

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight'); plt.show()

device: cuda | outputs -> /content/csi_results


## 2. Load data

In [8]:
import os
import h5py
from scipy.io import loadmat
import numpy as np

# Quick sanity check on file existence and size
if os.path.exists(DATA_PATH):
    file_size_mb = os.path.getsize(DATA_PATH) / (1024 * 1024)
    print(f"File '{DATA_PATH}' found. Size: {file_size_mb:.2f} MB")
    if file_size_mb < 1.0:
        print("Warning: The file size is very small. It might be a placeholder or corrupted link.")
else:
    print(f"Error: File '{DATA_PATH}' does not exist!")

try:
    # Try loading with standard scipy.io.loadmat
    print("Attempting to load with standard scipy.io.loadmat...")
    d = loadmat(DATA_PATH)
    H = d['H'].astype(np.complex64)
    pos = d['pos'].astype(np.float32)[:, :2]
    bs = d['bs_pos'].ravel()[:2]
    print("Successfully loaded using standard scipy.io.loadmat!")
except OSError as e:
    print("\n[CRITICAL ERROR] Standard loadmat failed to decompress file bytes.")
    print("Checking if it is a MATLAB v7.3 (HDF5) file...")
    try:
        # Try loading with h5py for v7.3 MAT files
        with h5py.File(DATA_PATH, 'r') as f:
            H = np.array(f['H']).astype(np.complex64)
            if H.ndim == 3 and H.shape[0] == 32 and H.shape[1] == 32:
                H = np.transpose(H, (2, 1, 0))
            pos = np.array(f['pos']).T.astype(np.float32)[:, :2]
            bs = np.array(f['bs_pos']).ravel()[:2]
        print("Successfully loaded using h5py!")
    except Exception as h5_err:
        print("\n[DIAGNOSIS] Both scipy.io.loadmat and h5py failed to parse the file structure.")
        print("This confirms the file 'csi_o1.mat' is either corrupted, truncated, or is not a valid MAT file.")
        print("Please delete the file, re-download/re-upload it completely, and try again.\n")
        raise e

N, N_ANT, N_SUB = H.shape
print('H', H.shape, '| pos', pos.shape, '| gNB at', bs)

File '/content/csi_o1.mat' found. Size: 208.00 MB
Attempting to load with standard scipy.io.loadmat...

[CRITICAL ERROR] Standard loadmat failed to decompress file bytes.
Checking if it is a MATLAB v7.3 (HDF5) file...

[DIAGNOSIS] Both scipy.io.loadmat and h5py failed to parse the file structure.
This confirms the file 'csi_o1.mat' is either corrupted, truncated, or is not a valid MAT file.
Please delete the file, re-download/re-upload it completely, and try again.



OSError: could not read bytes

## 3. Spatial split: whole 10 m blocks go to train / val / test

In [ ]:
# Neighbouring positions have almost identical CSI; a random split would put near-copies
# in train and test. Holding out whole blocks tests generalisation to unseen places.
block = np.floor((pos[:, 1] - pos[:, 1].min()) / BLOCK_M).astype(int)
split = np.full(N, 'train', dtype=object)
split[block % 5 == 2] = 'test'
split[block % 5 == 4] = 'val'
idx = {s: np.where(split == s)[0] for s in ['train', 'val', 'test']}
print({s: len(v) for s, v in idx.items()})

COLORS = {'train': 'tab:blue', 'val': 'tab:orange', 'test': 'tab:red'}
fig, ax = plt.subplots(figsize=(4, 6))
for s, c in COLORS.items():
    ax.scatter(pos[idx[s], 0], pos[idx[s], 1], s=0.3, c=c)
ax.plot(*bs, 'k^', ms=9)
handles = [Line2D([], [], marker='s', ls='', color=c, ms=8, label=s) for s, c in COLORS.items()]
handles.append(Line2D([], [], marker='^', ls='', color='k', ms=8, label='gNB (t006)'))
ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_title(f'Spatial split ({BLOCK_M} m blocks)')
ax.set_aspect('equal'); savefig('split.png')

## 4. Input representations (the only thing that differs between the two studies)

In [ ]:
# Same global scale for both: one constant computed on the training set, so a far user stays
# weaker than a near one and received power keeps carrying distance information.
CSI_SCALE = float(np.sqrt(np.mean(np.abs(H[idx['train']]) ** 2)))
POS_MEAN = pos[idx['train']].mean(0); POS_STD = pos[idx['train']].std(0)

def encode(Hc, rep):
    """complex (n, 32, 32) -> float tensor (n, 2, 32, 32)"""
    Hn = Hc / CSI_SCALE
    if rep == 'amp_phase':
        ch = [np.abs(Hn), np.angle(Hn) / np.pi]      # amplitude, phase scaled to [-1, 1]
    elif rep == 're_im':
        ch = [Hn.real, Hn.imag]
    else:
        raise ValueError(rep)
    return torch.from_numpy(np.stack(ch, 1).astype(np.float32))

def to_target(p):   return torch.from_numpy((p - POS_MEAN) / POS_STD)
def from_target(t): return t * POS_STD + POS_MEAN

X = {rep: {s: encode(H[i], rep) for s, i in idx.items()} for rep in REPS}
Y = {s: to_target(pos[i]) for s, i in idx.items()}

# What the network sees, for a near and a far user, in both representations
dist = np.hypot(pos[:, 0] - bs[0], pos[:, 1] - bs[1])
users = [int(np.argmin(np.abs(dist - 20))), int(np.argmin(np.abs(dist - 150)))]
fig, axs = plt.subplots(2, 4, figsize=(13, 6.2))
for r, u in enumerate(users):
    Hn = H[u] / CSI_SCALE
    for c, (img, title, cmap) in enumerate([
            (np.abs(Hn), '|H|', 'viridis'), (np.angle(Hn), 'phase (rad)', 'twilight'),
            (Hn.real, 'Re(H)', 'RdBu_r'), (Hn.imag, 'Im(H)', 'RdBu_r')]):
        im = axs[r, c].imshow(img, cmap=cmap, aspect='equal')
        axs[r, c].set_title(f'{title}, user at {dist[u]:.0f} m', fontsize=9)
        axs[r, c].set_xlabel('subcarrier'); axs[r, c].set_ylabel('antenna')
        cb = fig.colorbar(im, ax=axs[r, c], fraction=0.046)
        cb.formatter.set_useOffset(False); cb.update_ticks()
fig.suptitle('Amp/Phase (left pair) vs Re/Im (right pair): same information, different encoding')
savefig('representations.png')

## 5. Models

In [ ]:
class CNN(nn.Module):
    def __init__(self, in_ch=2):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                                            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU())
        self.f = nn.Sequential(blk(in_ch, 32), nn.MaxPool2d(2),   # 32 x 16 x 16
                               blk(32, 64), nn.MaxPool2d(2),      # 64 x 8 x 8
                               blk(64, 128), nn.AdaptiveAvgPool2d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 2))
    def forward(self, x): return self.head(self.f(x))

class ViT(nn.Module):
    """4x4 patches -> 64 tokens. Positional embedding factorised along the antenna (angle)
    and subcarrier (delay) axes, adapted to the structure of the CSI matrix."""
    def __init__(self, in_ch=2, n_ant=32, n_sub=32, pa=4, ps=4, dim=128, depth=4, heads=4, mlp=256, drop=0.1):
        super().__init__()
        self.patch = nn.Conv2d(in_ch, dim, kernel_size=(pa, ps), stride=(pa, ps))
        self.pos_ant = nn.Parameter(torch.zeros(1, dim, n_ant // pa, 1))
        self.pos_sub = nn.Parameter(torch.zeros(1, dim, 1, n_sub // ps))
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        for p in (self.pos_ant, self.pos_sub, self.cls): nn.init.trunc_normal_(p, std=.02)
        layer = nn.TransformerEncoderLayer(dim, heads, mlp, drop, activation='gelu',
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 2)
    def forward(self, x):
        z = self.patch(x) + self.pos_ant + self.pos_sub           # (B, dim, 8, 8)
        z = z.flatten(2).transpose(1, 2)                          # (B, 64, dim)
        z = torch.cat([self.cls.expand(len(z), -1, -1), z], 1)    # (B, 65, dim)
        return self.head(self.norm(self.enc(z)[:, 0]))

BUILD = {'CNN': CNN, 'ViT': ViT}
def n_params(m): return sum(p.numel() for p in m.parameters())
print('CNN params:', n_params(CNN()), '| ViT params:', n_params(ViT()))

## 6. Training and evaluation helpers

In [ ]:
def predict(model, Xs, bs=2048):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(Xs), bs):
            out.append(model(Xs[i:i + bs].to(device)).cpu())
    return torch.cat(out).numpy()

def errors_m(model, Xs, true_pos):
    """Euclidean distance (metres) between predicted and true positions."""
    return np.linalg.norm(from_target(predict(model, Xs)) - true_pos, axis=1)

def train(name, arch, Xtr, Ytr, Xval, pos_val, epochs, lr, seed):
    seed_all(seed)
    model = BUILD[arch]().to(device)
    Xtr, Ytr = Xtr.to(device), Ytr.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    steps = epochs * int(np.ceil(len(Xtr) / BATCH))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    loss_fn = nn.SmoothL1Loss(beta=0.05)
    best, best_state, hist, t0 = 1e9, None, [], time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr), device=device); tot = 0.0
        for i in range(0, len(Xtr), BATCH):
            b = perm[i:i + BATCH]
            loss = loss_fn(model(Xtr[b]), Ytr[b])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(b)
        val_err = errors_m(model, Xval, pos_val).mean()
        hist.append((tot / len(Xtr), val_err))
        if val_err < best:
            best, best_state = val_err, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'[{name}] {time.time()-t0:.0f}s | best val mean err {best:.2f} m (epoch {int(np.argmin([h[1] for h in hist]))})')
    return model, np.array(hist)

def summary(e):
    return dict(mean=float(e.mean()), median=float(np.median(e)),
                p90=float(np.percentile(e, 90)), rmse=float(np.sqrt((e ** 2).mean())))

def inference_ms(model, Xs, n=256):
    model.eval(); xb = Xs[:n].to(device)
    with torch.no_grad():
        model(xb)
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(10): model(xb)
        if device == 'cuda': torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / 10 / n      # ms per sample

## 7. Main experiment: 2 representations x 2 models x seeds

In [ ]:
runs = {}            # (rep, arch, seed) -> dict(model, hist, err)
for rep in REPS:
    for arch in MODELS:
        for s in SEEDS:
            m, h = train(f'{REP_LABEL[rep]} {arch} seed{s}', arch, X[rep]['train'], Y['train'],
                         X[rep]['val'], pos[idx['val']], EPOCHS[arch], LR[arch], s)
            runs[(rep, arch, s)] = dict(model=m, hist=h, err=errors_m(m, X[rep]['test'], pos[idx['test']]))

## 8. Results table (mean +- std over seeds, test zones)

In [ ]:
results = {}
print(f"{'input':10s} {'model':5s} {'mean (m)':>14s} {'median (m)':>14s} {'P90 (m)':>14s} {'RMSE (m)':>14s} {'params':>8s} {'ms/sample':>9s}")
for rep in REPS:
    for arch in MODELS:
        S = [summary(runs[(rep, arch, s)]['err']) for s in SEEDS]
        agg = {k: (float(np.mean([x[k] for x in S])), float(np.std([x[k] for x in S]))) for k in S[0]}
        m0 = runs[(rep, arch, SEEDS[0])]['model']
        agg['params'] = n_params(m0); agg['ms_per_sample'] = inference_ms(m0, X[rep]['test'])
        agg['per_seed'] = S
        results[f'{rep}/{arch}'] = agg
        f = lambda k: f"{agg[k][0]:6.2f} ± {agg[k][1]:4.2f}"
        print(f"{REP_LABEL[rep]:10s} {arch:5s} {f('mean'):>14s} {f('median'):>14s} {f('p90'):>14s} {f('rmse'):>14s} "
              f"{agg['params']:8,d} {agg['ms_per_sample']:9.3f}")

# Best seed-0 models are used for the figures below
STYLE = {('amp_phase', 'CNN'): ('tab:blue', '--'), ('amp_phase', 'ViT'): ('tab:orange', '--'),
         ('re_im', 'CNN'): ('tab:blue', '-'),      ('re_im', 'ViT'): ('tab:orange', '-')}
s0 = SEEDS[0]
def lab(rep, arch): return f'{arch} {REP_LABEL[rep]}'

## 9. CDF (zoomed), outliers, learning curves

In [ ]:
plt.figure(figsize=(5.5, 4))
for (rep, arch), (c, ls) in STYLE.items():
    e = np.sort(runs[(rep, arch, s0)]['err'])
    plt.plot(e, np.arange(1, len(e) + 1) / len(e), ls, c=c, label=lab(rep, arch))
plt.xlim(0, 15); plt.xlabel('Localization error (m)'); plt.ylabel('CDF')
plt.grid(alpha=.3); plt.legend(fontsize=8); plt.title('Error CDF, test zones (zoom 0-15 m)')
savefig('cdf_zoom.png')

print('Gross errors (> 50 m), seed', s0)
outliers = {}
for (rep, arch) in STYLE:
    m = runs[(rep, arch, s0)]['model']
    pred = from_target(predict(m, X[rep]['test'])); true = pos[idx['test']]
    e = runs[(rep, arch, s0)]['err']; bad = e > 50
    mirror = (np.sign(pred[bad, 1] - bs[1]) != np.sign(true[bad, 1] - bs[1])) if bad.any() else np.array([])
    far = np.abs(true[bad, 1] - bs[1]).mean() if bad.any() else float('nan')
    outliers[f'{rep}/{arch}'] = dict(n=int(bad.sum()), pct=float(100 * bad.mean()),
                                     wrong_side_pct=float(100 * mirror.mean()) if bad.any() else 0.0,
                                     mean_dist_to_gnb=float(far))
    print(f"  {lab(rep, arch):14s}: {bad.sum():3d} ({100*bad.mean():.2f}%), "
          f"{outliers[f'{rep}/{arch}']['wrong_side_pct']:.0f}% on the wrong side of the gNB, "
          f"mean |y - y_gNB| {far:.0f} m")

plt.figure(figsize=(5.5, 4))
for (rep, arch), (c, ls) in STYLE.items():
    plt.plot(runs[(rep, arch, s0)]['hist'][:, 1], ls, c=c, label=lab(rep, arch))
plt.xlabel('Epoch'); plt.ylabel('Validation mean error (m)'); plt.yscale('log')
plt.grid(alpha=.3); plt.legend(fontsize=8); plt.title('Learning curves (validation zones)')
savefig('learning_curves.png')

## 10. Error map: where does each model fail?

In [ ]:
allerr = np.concatenate([runs[(r, a, s0)]['err'] for (r, a) in STYLE])
vmax = np.percentile(allerr, 95)
fig, axs = plt.subplots(1, 4, figsize=(11, 6), sharey=True)
for ax, (rep, arch) in zip(axs, [('amp_phase', 'CNN'), ('amp_phase', 'ViT'), ('re_im', 'CNN'), ('re_im', 'ViT')]):
    sc = ax.scatter(pos[idx['test'], 0], pos[idx['test'], 1], c=runs[(rep, arch, s0)]['err'],
                    s=1, cmap='viridis', vmax=vmax)
    ax.plot(*bs, 'r^', ms=8); ax.set_title(lab(rep, arch), fontsize=9); ax.set_xlabel('x (m)')
axs[0].set_ylabel('y (m)')
fig.colorbar(sc, ax=axs, label=f'error (m), capped at {vmax:.1f}')
plt.savefig(os.path.join(FIG_DIR, 'error_map.png'), dpi=200, bbox_inches='tight'); plt.show()

## 11. Robustness to noise (complex AWGN added to the test CSI, then encoded; no retraining)

In [ ]:
SNRS = [0, 5, 10, 15, 20, 30]
rng = np.random.default_rng(0)
Ht = H[idx['test']]
p_sig = np.mean(np.abs(Ht) ** 2, axis=(1, 2), keepdims=True)
snr_res = {f'{r}/{a}': [] for (r, a) in STYLE}
for snr in SNRS:
    noise = (rng.standard_normal(Ht.shape) + 1j * rng.standard_normal(Ht.shape)) / np.sqrt(2)
    Hn = (Ht + np.sqrt(p_sig / 10 ** (snr / 10)) * noise).astype(np.complex64)
    for rep in REPS:
        Xn = encode(Hn, rep)
        for arch in MODELS:
            snr_res[f'{rep}/{arch}'].append(float(errors_m(runs[(rep, arch, s0)]['model'], Xn, pos[idx['test']]).mean()))
plt.figure(figsize=(5.5, 4))
for (rep, arch), (c, ls) in STYLE.items():
    plt.plot(SNRS, snr_res[f'{rep}/{arch}'], 'o' + ls, c=c, label=lab(rep, arch))
plt.xlabel('SNR (dB)'); plt.ylabel('Mean error (m)'); plt.yscale('log')
plt.grid(alpha=.3); plt.legend(fontsize=8); plt.title('Robustness to noise (test-time AWGN)')
savefig('snr.png')

## 12. Training-set size (25 / 50 / 100 % of the training set, seed 0)

In [ ]:
size_res = {}
if RUN_DATA_SIZE:
    fracs = [0.25, 0.5, 1.0]
    g = torch.Generator().manual_seed(0)
    perm = torch.randperm(len(Y['train']), generator=g)
    for rep in REPS:
        for arch in MODELS:
            key = f'{rep}/{arch}'; size_res[key] = []
            for f in fracs:
                if f == 1.0:
                    size_res[key].append(float(runs[(rep, arch, s0)]['err'].mean())); continue
                sub = perm[:int(f * len(perm))]
                m, _ = train(f'{lab(rep, arch)} {int(f*100)}%', arch, X[rep]['train'][sub], Y['train'][sub],
                             X[rep]['val'], pos[idx['val']], EPOCHS[arch], LR[arch], s0)
                size_res[key].append(float(errors_m(m, X[rep]['test'], pos[idx['test']]).mean()))
    n_tr = [int(f * len(perm)) for f in fracs]
    plt.figure(figsize=(5.5, 4))
    for (rep, arch), (c, ls) in STYLE.items():
        plt.plot(n_tr, size_res[f'{rep}/{arch}'], 'o' + ls, c=c, label=lab(rep, arch))
    plt.xlabel('Training samples'); plt.ylabel('Mean test error (m)')
    plt.grid(alpha=.3); plt.legend(fontsize=8); plt.title('Effect of training-set size')
    savefig('data_size.png')

## 13. CNN learning-rate check (is the CNN baseline fairly tuned?)

In [ ]:
cnn_lr = {}
if RUN_CNN_LR_CHECK:
    for rep in REPS:
        m, h = train(f'{REP_LABEL[rep]} CNN lr=1e-3', 'CNN', X[rep]['train'], Y['train'],
                     X[rep]['val'], pos[idx['val']], EPOCHS['CNN'], 1e-3, s0)
        e = errors_m(m, X[rep]['test'], pos[idx['test']])
        cnn_lr[rep] = {'lr_2e-3': summary(runs[(rep, 'CNN', s0)]['err']), 'lr_1e-3': summary(e)}
        print(f"{REP_LABEL[rep]} CNN: mean {cnn_lr[rep]['lr_2e-3']['mean']:.2f} m (lr 2e-3) "
              f"vs {cnn_lr[rep]['lr_1e-3']['mean']:.2f} m (lr 1e-3)")

## 14. Save everything for the article

In [ ]:
out = dict(config=dict(reps=REPS, seeds=SEEDS, block_m=BLOCK_M, epochs=EPOCHS, lr=LR, batch=BATCH,
                       split={s: int(len(v)) for s, v in idx.items()}, device=device),
           results=results, outliers=outliers, snr_db=SNRS, snr_mean_err=snr_res,
           data_size_mean_err=size_res, cnn_lr_check=cnn_lr)
with open(os.path.join(OUT_DIR, 'results.json'), 'w') as fh:
    json.dump(out, fh, indent=2)
torch.save({f'{r}/{a}': runs[(r, a, s0)]['model'].state_dict() for (r, a) in STYLE},
           os.path.join(OUT_DIR, 'models.pt'))
print('Saved results.json, models.pt and figures to', OUT_DIR)